In [5]:
#filtering cells using regionprops and extract intensity infromation from the raw image
# Import necessary libraries

import numpy as np
from scipy.ndimage import label
from skimage import measure
from scipy.spatial import distance
from scipy.ndimage import zoom
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
import time
import csv
import re
import os
import pandas as pd
from skimage.measure import regionprops
import tifffile as tif

def extract_odorant(filename):
    pattern = r'(?i)CaMPARI([A-Za-z0-9]+)'  # Added (?i) for case-insensitive matching
    match = re.search(pattern, filename)
    return match.group(1) if match else None
def processed_output_exists(filename, output_folder):
    # Check if the corresponding CSV output exists in the output folder.
    odorant_name = extract_odorant(filename)
    csv_filename = os.path.join(output_folder, f'{odorant_name}_cell_distances_intensities.csv')
    return os.path.exists(csv_filename)

def filter_cells_by_distance(cell_segmentation_image, labeled_image, raw_data_image, filename, pixel_to_micron, output_folder,distance):
    start_time = time.time()
    selected_labels = {1: "mdG2", 2: "maG", 3: "mdG6", 4: "dG", 5: "vmG", 6: "lG", 7: "dlG", 8: "vpG"}
#    selected_labels = {1: "mdG2", 2: "maG"}
    
    # Get unique cell labels
    unique_cell_labels = np.unique(cell_segmentation_image)
    unique_cell_labels = unique_cell_labels[unique_cell_labels != 0]  # Exclude background label if present

    # DataFrame to save cell centroid, distance to each label, and intensity measurements
    cell_data_df = pd.DataFrame(index=unique_cell_labels)
    cell_data_df.index.name = 'cell_label_id'
    
    # Dictionary for filtered cells images
    filtered_cells_images_by_label = {label_name: np.zeros_like(cell_segmentation_image) for label_name in selected_labels.values()}
    
    for label_value, label_name in selected_labels.items():
        region_mask = (labeled_image == label_value)
        distance_transform = distance_transform_edt(~region_mask)
        boundary_coords = np.argwhere(distance_transform == 1)
        boundary_coords_scaled = boundary_coords * [pixel_to_micron['z'], pixel_to_micron['y'], pixel_to_micron['x']]
        boundary_tree = cKDTree(boundary_coords_scaled)

        # Calculate region properties for centroids and volumes
        props = regionprops(label_image=cell_segmentation_image)
        centroids = []
        volumes = []
        for prop in props:
            if prop['label'] in unique_cell_labels:
                centroids.append(prop.centroid)
                volumes.append(prop.area)  # In 3D, 'area' corresponds to the volume
        
        # Ensure there are centroids found before proceeding
        if not centroids:
            print(f"No centroids found for file: {filename}. Skipping processing for this file.")
            return None, None
        
        centroids = np.array(centroids)
        volumes = np.array(volumes)
        
        # Scale centroids
        centroids_scaled = centroids * [pixel_to_micron['z'], pixel_to_micron['y'], pixel_to_micron['x']]
        
        # Save the scaled centroids to the DataFrame
        cell_data_df['centroid_x'] = centroids_scaled[:, 2]  # x-coordinate
        cell_data_df['centroid_y'] = centroids_scaled[:, 1]  # y-coordinate
        cell_data_df['centroid_z'] = centroids_scaled[:, 0]  # z-coordinate
        
        # Save volumes to the DataFrame
        cell_data_df['volume'] = volumes * np.prod(list(pixel_to_micron.values()))  # Adjust volume by pixel to micron scaling
        min_distances, _ = boundary_tree.query(centroids_scaled, k=1)

        # Save distances to current label
        cell_data_df[label_name] = pd.Series(min_distances, index=[prop['label'] for prop in props if prop['label'] in unique_cell_labels])

        # Filter cells by distance and update filtered_cells_images_by_label
        filtered_cells_labels = [props[i]['label'] for i in np.arange(len(props)) if min_distances[i] < distance]
        for i, cell_label in enumerate(filtered_cells_labels):
            filtered_cells_images_by_label[label_name][cell_segmentation_image == cell_label] = cell_label

        # Collect intensity statistics for each channel
        for channel in range(3):
            single_channel_intensity_image = raw_data_image[:, channel, :, :]
            props_intensity = regionprops(label_image=cell_segmentation_image, intensity_image=single_channel_intensity_image)
            mean_intensities = [prop['mean_intensity'] for prop in props_intensity]
            max_intensities = [prop['max_intensity'] for prop in props_intensity]
            min_intensities = [prop['min_intensity'] for prop in props_intensity]
            cell_data_df[f'mean_intensity_channel_{channel}'] = mean_intensities
            cell_data_df[f'max_intensity_channel_{channel}'] = max_intensities
            cell_data_df[f'min_intensity_channel_{channel}'] = min_intensities

    # Save to CSV
    odorant_name = extract_odorant(filename)
    csv_filename = os.path.join(output_folder, f'{odorant_name}_cell_distances_intensities.csv')
    cell_data_df.to_csv(csv_filename)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Filtering cells took {elapsed_time:.2f} seconds.")

    return filtered_cells_images_by_label, cell_data_df  # Return both values





In [10]:

def extract_odorant(filename):
    # Define a pattern to match the odorant information (letters and numbers following "CamPARI")
    pattern = r'(?i)CaMPARI([A-Za-z0-9]+)'  # Added (?i) for case-insensitive matching
    match = re.search(pattern, filename)
    return match.group(1) if match else None

def process_data(raw_data_folder, cell_segmentation_folder, labeled_image_path, output_folder, pixel_to_micron, distance):
    # Read the labeled image
    labeled_image = tif.imread(labeled_image_path)

    # Iterate through the raw data files
    for filename in os.listdir(raw_data_folder):
        if filename.endswith(".tif"):
            # Check if the file has already been processed
            if processed_output_exists(filename, output_folder):
                print(f'Skipping processing for {filename} as it has already been processed.')
                continue

            print('Processing', filename)
            raw_data_path = os.path.join(raw_data_folder, filename)
            cell_segmentation_filename = filename.replace('_merged.tif', '_merged_expanded_mask.tif')
            cell_segmentation_path = os.path.join(cell_segmentation_folder, cell_segmentation_filename)
            print(f' cell segmentation path {cell_segmentation_path}')
            # Read the raw data and cell segmentation images
            raw_data_image = tif.imread(raw_data_path)
            cell_segmentation_image = tif.imread(cell_segmentation_path)
            cell_segmentation_image = np.squeeze(cell_segmentation_image)

            # Filter cells by distance
            filtered_cells_images_by_label, cell_data_df = filter_cells_by_distance(cell_segmentation_image, labeled_image, raw_data_image, filename, pixel_to_micron, output_folder, distance)

    print("Processing completed.")
    return filtered_cells_images_by_label, cell_data_df, cell_segmentation_image, raw_data_image, labeled_image


# Define the paths and parameters
raw_data_folder = r"U:\VAMP\maysel0000\202312040506_campari"
cell_segmentation_folder = raw_data_folder + os.sep + "output/expanded_masks/"
#cell_segmentation_folder = r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20240116_CaMPARI_100uMCitricAcid_60sec_10secUV\tiff_files\reg_files\merged_files\output\expanded_masks"
output_folder = cell_segmentation_folder  + os.sep + "output"
if not os.path.exists(output_folder):
    os.mkdir(output_folder)
#output_folder = "U:/Home/maysel0000/imaging_data/processed_data/20240116_CaMPARI_100uMCitricAcid_60sec_10secUV/tiff_files/reg_files/merged_files/output/expanded_masks/output"
labeled_image_path = "U:\VAMP\maysel0000\campari_pipeline\glomeruli_as_onech.tif"
pixel_to_micron = {'x': 0.4333, 'y': 0.4333, 'z': 1.0}
distance = 5
# Run the process
filtered_cells_images_by_label, cell_data_df, cell_segmentation_image, raw_data_image, labeled_image = process_data(raw_data_folder, cell_segmentation_folder, labeled_image_path, output_folder, pixel_to_micron,distance)


Processing ref_20231204CaMPARIArg500uM60sec10secUV01_1_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
 cell segmentation path U:\VAMP\maysel0000\202312040506_campari\output/expanded_masks/ref_20231204CaMPARIArg500uM60sec10secUV01_1_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
Filtering cells took 967.93 seconds.
Processing ref_20231204CaMPARIArg500uM60sec10secUV02_2_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
 cell segmentation path U:\VAMP\maysel0000\202312040506_campari\output/expanded_masks/ref_20231204CaMPARIArg500uM60sec10secUV02_2_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
Filtering cells took 935.48 seconds.
Processing ref_20231204CaMPARIArg500uM60sec10secUV03_3_01_warp_m0g80c4e1e-1x52r3.nrrd_merged.tif
 cell segmentation path U:\VAMP\maysel0000\202312040506_campari\output/expanded_masks/ref_20231204CaMPARIArg500uM60sec10secUV03_3_01_warp_m0g80c4e1e-1x52r3.nrrd_merged_expanded_mask.tif
Filtering cells took 947.03 seconds.
Processing ref_20231204CaMPARIArg500uM60

In [4]:
import napari

# Create a Napari viewer
viewer = napari.Viewer()

# Add the original cell segmentation image as a layer
viewer.add_labels(cell_segmentation_image, name='Original Cell Segmentation')
viewer.add_labels(labeled_image, name="Labeled Glomeruli")
viewer.add_image(raw_data_image, name="raw_data",channel_axis=1)
# Add the filtered cells images for each label as separate layers
for label_name, filtered_cells_image in filtered_cells_images_by_label.items():
    viewer.add_labels(filtered_cells_image
                      , name=f'Filtered Cells - {label_name}')
viewer.dims.ndisplay = 3
# Display the Napari viewer
napari.run()


MemoryError: Unable to allocate 1.49 GiB for an array with shape (301, 1152, 1152) and data type float32